In [ ]:
!pip install transformers torch bertviz -q

from bertviz import head_view, model_view
from transformers import BertTokenizer, BertModel
import torch

In [ ]:
!pip install transformers torch bertviz -q

from bertviz import head_view
from transformers import BertTokenizer, BertModel, pipeline
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# CHARGEMENT FINBERT
# ============================================================
model_name = "ProsusAI/finbert"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_attentions=True)
model.eval()

# ============================================================
# PHRASE + TOKENISATION
# ============================================================
sentence = "The results are not disappointing"
inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"Phrase : '{sentence}'")
print(f"Tokens : {tokens}")

# ============================================================
# INFERENCE
# ============================================================
with torch.no_grad():
    outputs = model(**inputs)

attention = outputs.attentions
# 12 couches × 12 têtes × (seq_len × seq_len)

# ============================================================
# MATRICE D'ATTENTION — CHOISIR LA COUCHE ET LA TÊTE
# ============================================================
couche = 11  # ← changer ici (0 à 11)
tete   = 1  # ← changer ici (0 à 11)

attn_matrix = attention[couche][0][tete].detach().numpy()

print(f"\n📊 Matrice — Couche {couche+1}, Tête {tete+1}")
print(f"   Valeurs entre : {attn_matrix.min():.3f} et {attn_matrix.max():.3f}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(attn_matrix,
            xticklabels=tokens, yticklabels=tokens,
            cmap='YlOrRd', annot=True, fmt='.2f',
            vmin=0, vmax=1, ax=ax)
ax.set_title(f"Scores d'attention — Couche {couche+1}, Tête {tete+1}\n'{sentence}'")
ax.set_xlabel("Keys")
ax.set_ylabel("Queries")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# VISUALISATION BERTVIZ
head_view(attention, tokens, layer=couche, heads=[tete])

In [ ]:
nlp = pipeline("text-classification", model="ProsusAI/finbert")
result = nlp(sentence)[0]

print(f"\n Sentiment prédit : {result['label']}")
print(f"   Confiance : {round(result['score']*100, 2)}%")